# Latent Dirichlet Allocation (LDA) for Topic Modelling

https://radimrehurek.com/gensim/auto_examples/tutorials/run_lda.html

In [2]:
# imports

import mwxml
import re
import numpy as np
import pandas as pd
import spacy
from tqdm import tqdm

: 

: 

## Load Wikipedia Data

In [18]:
dump = mwxml.Dump.from_file(open("../Data/dewiki-latest-pages-articles-multistream.xml"))

def dump_to_dataset(dump:mwxml.iteration.dump.Dump, n_samples:int=-1) -> tuple([np.ndarray, np.ndarray]):
    data = []
    label = []
    i = 0
    #pbar = tqdm(total= (n_samples if (n_samples>1) else 50000000))
    for page in dump:
        for revisions in page:
            try:
                if("Liste von Autoren" not in revisions.page.title):
                    i += 1
                    if(re.search(r"{{Exzellent[|](\d*).(\D*)(\d*)[|](\d*)}}", revisions.text)):
                        label.append(1)
                        data.append(re.sub(r"{{Exzellent[|](\d*).(\D*)(\d*)[|](\d*)}}", '',revisions.text))
                    else:
                        pass
                    #pbar.update(1)
            except Exception as e:
                print(e)
        if(i>=n_samples):
            break
    #pbar.close()
    return np.array(data), np.array(label)

In [19]:
features, labels = dump_to_dataset(dump, 10000)

In [20]:
print(len(features), len(labels))

252 252


In [21]:
df = pd.DataFrame(features, columns=['feature1'])
df['label'] = labels
docs = df['feature1']
df

,feature1,label
0,{{Begriffsklärungshinweis}}\n{{Infobox Chemisc...,1
1,{{Dieser Artikel|behandelt das chemische Eleme...,1
2,{{Infobox Chemisches Element\n<!--- Periodensy...,1
3,{{Dieser Artikel|behandelt das naturwissenscha...,1
4,{{Begriffsklärungshinweis}}\n'''Aristoteles'''...,1
...,...,...
247,{{Infobox Gemeinde in Deutschland\n|Art ...,1
248,"{{Begriffsklärungshinweis}}\n{| class=""wikitab...",1
249,{{Weiterleitungshinweis|Baskisch}}\n{{Infobox ...,1
250,[[Datei:TSP Deutschland 3.png|mini|Optimaler R...,1


In [ ]:
# remove wikipedia specific stopwords / ref / name / Tabelle 

## Create tokenized corpus

In [7]:
# Tokenize the documents

from nltk.tokenize import RegexpTokenizer

# Split the documents into tokens

tokenizer = RegexpTokenizer(r'\w+')
for idx in range(len(docs)):
    docs[idx] = docs[idx].lower()  # Convert to lowercase
    docs[idx] = tokenizer.tokenize(docs[idx])  # Split into words

# Remove unnecessary words

docs = [[token for token in doc if not token.isnumeric()] for doc in docs] # Remove numbers

docs = [[token for token in doc if len(token) > 1] for doc in docs] # Remove words with only one character

/var/folders/2j/wfkfpv3d0399hnl84gpblcp40000gn/T/ipykernel_66879/887269700.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  docs[idx] = docs[idx].lower()  # Convert to lowercase
/var/folders/2j/wfkfpv3d0399hnl84gpblcp40000gn/T/ipykernel_66879/887269700.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  docs[idx] = tokenizer.tokenize(docs[idx])  # Split into words
/var/folders/2j/wfkfpv3d0399hnl84gpblcp40000gn/T/ipykernel_66879/887269700.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable

In [8]:
# Lemmatize the documents english

import nltk
from nltk.corpus import wordnet as wn
from nltk.stem.wordnet import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
docs = [[lemmatizer.lemmatize(token) for token in doc] for doc in docs]

In [2]:
# Lemmatize the documents in german

import spacy
nlp = spacy.load('de_core_news_md')

mails=['Hallo. Ich spielte am frühen Morgen und ging dann zu einem Freund. Auf Wiedersehen', 'Guten Tag Ich mochte Bälle und will etwas kaufen. Tschüss']

mails_lemma = []

for mail in mails:
     doc = nlp(mail)
     result = ' '.join([x.lemma_ for x in doc]) 
     mails_lemma.append(result)

OSError: [E050] Can't find model 'de_core_news_md'. It doesn't seem to be a Python package or a valid path to a data directory.

## Lemmatizizer

Während ein Stemmer Wörter auf ihren Stamm reduziert, indem die Endungen entfernt werden, wird ein Lemmatizer eingesetzt um Wörter in ihre tatsächliche Grundform zu bringt. Zwar ist der Stemmer einfacher und schneller, doch in diesem Fall sollen die Wörter weiterhin ein gramatikalisch sundvolles Wort ergeben, welches als Keyword verwendet werden kann. Aus Experte und Expertin soll nicht Expert (Stemmer) sonder Experte (Lemmatizer) werden.

In [22]:
docs = docs[:3]

In [23]:
from HanTa import HanoverTagger as ht

tagger = ht.HanoverTagger('morphmodel_ger.pgz')

mails=['Hallo. Ich spielte am frühen Morgen und ging dann zu einem Freund. Auf Wiedersehen',
       'Guten Tag Ich mochte Bälle und will etwas kaufen. Tschüss']

mails_lemma = []
for mail in docs:
    lemma = [lemma for (word,lemma,pos) in tagger.tag_sent(mail.split())]
    mails_lemma.append(' '.join(lemma))

mails_lemma

['{{begriffsklärungshinweis}} {{infobox chemisch Element <!--- Periodensystem ---> | Name = Argon | Symbol = ar | Ordnungszahl = 18 | Serie = Eg | Gruppe = 18 | Periode = 3 | Block = p | Hauptquelle = <ref Name="webelements">die Wert für der Eigenschaft (infobox) Sind, wenn nicht anders Angegeben, aus [http://www.webelements.com/argon/ www.webelements.com (argon)] Entnommen.</ref> <!--- allgemein ---> | Aussehen = farblos Gas | Massenanteil = 3,6&nbsp;[[parts per Million|ppm]] (53. Rang)<ref Name="harry H. Binder">harry H. Binder: \'\'lexikon der chemisch Elemente.\'\' S. Hirzel Verlag, Stuttgart 1999, Isbn 3-7776-0736-3.</ref> | cas = {{casrn|7440-37-1}} | Eg-nummer = 231-147-0 | echa-id = 100.028.315 <!--- atomar ---> | Atommasse = 39,948 (39,792-39,963)<ref>angegebe sein der von der Iupac empfohlen Standardwert, da der Isotopenzusammensetzung dieser Element örtlich schwanken Kann, ergeben sich für der mittler Atomgewicht der in Klammer angegeben Massenbereich. Siehe: Iupac Commissio

## Remove Wikipedia spfcific stopwords

https://medium.com/@tusharsri/remove-add-stop-words-7e2994c19c67

In [7]:
import spacy 
from spacy.lang.de.stop_words import STOP_WORDS

print(len(STOP_WORDS))
print(STOP_WORDS)

543
{'dessen', 'war', 'einander', 'gross', 'solches', 'kleinen', 'oben', 'ag', 'aber', 'auf', 'wohl', 'aus', 'deiner', 'geht', 'achtes', 'siebte', 'achten', 'wer', 'seien', 'rechten', 'eigener', 'kommt', 'her', 'rechter', 'diesem', 'mussten', 'eines', 'ganz', 'demselben', 'na', 'wart', 'wessen', 'dem', 'grosser', 'lang', 'durch', 'anders', 'ins', 'über', 'andern', 'guter', 'ohne', 'fünfte', 'willst', 'kaum', 'gute', 'bei', 'darfst', 'deshalb', 'bis', 'tagen', 'unserer', 'elf', 'euch', 'sind', 'achte', 'demgemäss', 'ja', 'ganzen', 'dich', 'seitdem', 'solche', 'leicht', 'musste', 'richtig', 'hatte', 'habe', 'beispiel', 'konnte', 'sagte', 'heisst', 'lange', 'manchen', 'dabei', 'weniger', 'jeden', 'bereits', 'sein', 'welche', 'sie', 'los', 'wie', 'siebente', 'jemandem', 'sechsten', 'denen', 'denn', 'dazwischen', 'seines', 'fünftes', 'jenem', 'ende', 'jedoch', 'macht', 'zwischen', 'durfte', 'ihre', 'diesen', 'wegen', 'zunächst', 'wurden', 'nur', 'eben', 'kein', 'er', 'diejenigen', 'jede', '

In [6]:
import spacy
nlp = spacy.load('en_core_web_sm')

sentence = nlp("We will go to movie after the dinner")
print(sentence)

notStopWords = [notStopWords.text for notStopWords in sentence if not notStopWords.is_stop]
print(notStopWords)

stopWords = [stopWords.text for stopWords in sentence if stopWords.is_stop]
print(stopWords)

We will go to movie after the dinner
['movie', 'dinner']
['We', 'will', 'go', 'to', 'after', 'the']


In [ ]:
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer


german_stop_words = stopwords.words('german')

vect = CountVectorizer(stop_words = german_stop_words) # Now use this in your pipeline

In [9]:
# Compute bigrams
from gensim.models import Phrases

# Add bigrams and trigrams to docs (only ones that appear 20 times or more)

bigram = Phrases(docs, min_count=20)
for idx in range(len(docs)):
    for token in bigram[docs[idx]]:
        if '_' in token:
            # Token is a bigram, add to document
            docs[idx].append(token)

In [10]:
# Remove rare and common tokens

from gensim.corpora import Dictionary

# Create a dictionary representation of the documents

dictionary = Dictionary(docs)

# Filter out words that occur less than 20 documents, or more than 50% of the documents

# dictionary.filter_extremes(no_below=20, no_above=0.5)

In [11]:
# Bag-of-words representation of the documents

corpus = [dictionary.doc2bow(doc) for doc in docs]

In [12]:
print('Number of unique tokens: %d' % len(dictionary))
print('Number of documents: %d' % len(corpus))

Number of unique tokens: 179193
Number of documents: 252


# Initialize and train LDA Model

In [13]:
from gensim.models import LdaModel

# Set training parameters
num_topics = 10
chunksize = 2000
passes = 20
iterations = 400
eval_every = None  # Don't evaluate model perplexity, takes too much time.

# Make an index to word dictionary
temp = dictionary[0] 
id2word = dictionary.id2token

model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    chunksize=chunksize,
    alpha='auto',
    eta='auto',
    iterations=iterations,
    num_topics=num_topics,
    passes=passes,
    eval_every=eval_every
)

## LDA2VEC

In [ ]:
# Test

In [2]:
! pip install git+https://github.com/LIAAD/yake

  Cloning https://github.com/LIAAD/yake to /private/var/folders/2j/wfkfpv3d0399hnl84gpblcp40000gn/T/pip-req-build-2e13oi7u
  Running command git clone --filter=blob:none --quiet https://github.com/LIAAD/yake /private/var/folders/2j/wfkfpv3d0399hnl84gpblcp40000gn/T/pip-req-build-2e13oi7u
  Resolved https://github.com/LIAAD/yake to commit 374fc1c1c19eb080d5b6115cbb8d4a4324392e54
  Preparing metadata (setup.py) ... done
  Using cached segtok-1.5.11-py3-none-any.whl (24 kB)
  Using cached networkx-3.1-py3-none-any.whl (2.1 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.9/328.9 kB 3.3 MB/s eta 0:00:00a 0:00:01
  Created wheel for yake: filename=yake-0.4.8-py2.py3-none-any.whl size=62574 sha256=0c03e65aa36abf874126d76da1161fbf74e930002149c142188a4e65b89fd647
  Stored in directory: /private/var/folders/2j/wfkfpv3d0399hnl84gpblcp40000gn/T/pip-ephem-wheel-cache-63c91vfy/wheels/06/e6/1c/8f614adcd4b053020d672b9cbc5ef1166407755a4b71adea0d
Successfully built yake


## YAKE

In [25]:
import yake

text = "Sources tell us that Google is acquiring Kaggle, a platform that hosts data science and machine learning "\
"competitions. Details about the transaction remain somewhat vague, but given that Google is hosting its Cloud "\
"Next conference in San Francisco this week, the official announcement could come as early as tomorrow. "\
"Reached by phone, Kaggle co-founder CEO Anthony Goldbloom declined to deny that the acquisition is happening. "\
"Google itself declined 'to comment on rumors'. Kaggle, which has about half a million data scientists on its platform, "\
"was founded by Goldbloom  and Ben Hamner in 2010. "\
"The service got an early start and even though it has a few competitors like DrivenData, TopCoder and HackerRank, "\
"it has managed to stay well ahead of them by focusing on its specific niche. "\
"The service is basically the de facto home for running data science and machine learning competitions. "\
"With Kaggle, Google is buying one of the largest and most active communities for data scientists - and with that, "\
"it will get increased mindshare in this community, too (though it already has plenty of that thanks to Tensorflow "\
"and other projects). Kaggle has a bit of a history with Google, too, but that's pretty recent. Earlier this month, "\
"Google and Kaggle teamed up to host a $100,000 machine learning competition around classifying YouTube videos. "\
"That competition had some deep integrations with the Google Cloud Platform, too. Our understanding is that Google "\
"will keep the service running - likely under its current name. While the acquisition is probably more about "\
"Kaggle's community than technology, Kaggle did build some interesting tools for hosting its competition "\
"and 'kernels', too. On Kaggle, kernels are basically the source code for analyzing data sets and developers can "\
"share this code on the platform (the company previously called them 'scripts'). "\
"Like similar competition-centric sites, Kaggle also runs a job board, too. It's unclear what Google will do with "\
"that part of the service. According to Crunchbase, Kaggle raised $12.5 million (though PitchBook says it's $12.75) "\
"since its   launch in 2010. Investors in Kaggle include Index Ventures, SV Angel, Max Levchin, Naval Ravikant, "\
"Google chief economist Hal Varian, Khosla Ventures and Yuri Milner "

In [26]:
import yake

language = "de"
max_ngram_size = 1
deduplication_threshold = 0.9
deduplication_algo = 'seqm'
windowSize = 1
numOfKeywords = 3


for i in docs :
    custom_kw_extractor = yake.KeywordExtractor(lan=language, n=max_ngram_size, dedupLim=deduplication_threshold, dedupFunc=deduplication_algo, windowsSize=windowSize, top=numOfKeywords, features=None)
    keywords = custom_kw_extractor.extract_keywords(i)

    for kw in keywords:
        print(kw)

('sup', 0.0011520623785751514)
('Argon', 0.0019904444618167264)
('ref', 0.003333483979903347)
('Arsen', 0.00030341031732455015)
('sub', 0.00040182535584515464)
('ref', 0.0010476990366250995)
('sup', 0.0005480921504543687)
('Americium', 0.000805594410056688)
('sub', 0.0009814906472720175)


In [11]:
kw_extractor = yake.KeywordExtractor()
keywords = kw_extractor.extract_keywords(text)

for kw in keywords:
	print(kw)

('Google', 0.026580863364597897)
('Kaggle', 0.0289005976239829)
('CEO Anthony Goldbloom', 0.029946071606210194)
('San Francisco', 0.048810837074825336)
('Anthony Goldbloom declined', 0.06176910090701819)
('Google Cloud Platform', 0.06261974476422487)
('co-founder CEO Anthony', 0.07357749587020043)
('acquiring Kaggle', 0.08723571551039863)
('CEO Anthony', 0.08915156857226395)
('Anthony Goldbloom', 0.09123482372372106)
('machine learning', 0.09147989238151344)
('Kaggle co-founder CEO', 0.093805063905847)
('data', 0.097574333771058)
('Google Cloud', 0.10260128641464673)
('machine learning competitions', 0.10773000650607861)
('Francisco this week', 0.11519915079240485)
('platform', 0.1183512305596321)
('conference in San', 0.12392066376108138)
('service', 0.12546743261462942)
('Goldbloom', 0.14611408778815776)


In [12]:
kw_extractor = yake.KeywordExtractor()
keywords = kw_extractor.extract_keywords(docs[0])

for kw in keywords:
	print(kw)

Warning! Exception: 'list' object has no attribute 'replace' generated by the following text: '['begriffsklärungshinweis', 'infobox', 'chemisches', 'element', 'periodensystem', 'name', 'argon', 'symbol', 'ar', 'ordnungszahl', 'serie', 'eg', 'gruppe', 'periode', 'block', 'hauptquelle', 'ref', 'name', 'webelements', 'die', 'werte', 'für', 'die', 'eigenschaften', 'infobox', 'sind', 'wenn', 'nicht', 'anders', 'angegeben', 'aus', 'http', 'www', 'webelements', 'com', 'argon', 'www', 'webelements', 'com', 'argon', 'entnommen', 'ref', 'allgemein', 'aussehen', 'farbloses', 'gas', 'massenanteil', 'nbsp', 'parts', 'per', 'million', 'ppm', 'rang', 'ref', 'name', 'harry', 'binder', 'harry', 'binder', 'lexikon', 'der', 'chemischen', 'elemente', 'hirzel', 'verlag', 'stuttgart', 'isbn', 'ref', 'cas', 'casrn', 'eg', 'nummer', 'echa', 'id', 'atomar', 'atommasse', 'ref', 'angegeben', 'ist', 'der', 'von', 'der', 'iupac', 'empfohlene', 'standardwert', 'da', 'die', 'isotopenzusammensetzung', 'dieses', 'elem

# Get top 3 Keywords

In [14]:
keywords_list = []

for i in range (0, len(corpus)):
    topic_distribution = model.get_document_topics(corpus[i])
    # Sort the topics by their probability in descending order
    sorted_topics = sorted(topic_distribution, key=lambda x: x[1], reverse=True)
    # Get the top N keywords for the highest-probability topic
    top_keywords = model.show_topic(sorted_topics[0][0], topn=3)
    # Extract and return the keyword strings
    keywords = [keyword for keyword, _ in top_keywords]

    keywords_list.append(keywords)

print(keywords_list)

[['Die', 'In', 'Der'], ['Die', 'In', 'ISBN'], ['Titanic', 'Die', 'Tower'], ['Die', 'In', 'ISBN'], ['Die', 'In', 'Der'], ['Die', 'In', 'Der'], ['Die', 'In', 'Der'], ['Die', 'In', 'Der'], ['Die', 'In', 'Der'], ['Die', 'In', 'Der'], ['Die', 'Der', 'In'], ['Die', 'Dresden', 'In'], ['Titanic', 'Die', 'Tower'], ['Die', 'In', 'ISBN'], ['Die', 'In', 'ISBN'], ['Die', 'In', 'Der'], ['Die', 'Bonn', 'Der'], ['Die', 'In', 'ISBN'], ['Die', 'In', 'Der'], ['Die', 'In', 'ISBN'], ['Die', 'In', 'ISBN'], ['Titanic', 'Die', 'Tower'], ['Titanic', 'Die', 'Tower'], ['Die', 'In', 'Der'], ['Die', 'Goethe', 'In'], ['Die', 'In', 'Der'], ['Die', 'In', 'Der'], ['Die', 'Der', 'In'], ['Die', 'In', 'Der'], ['Die', 'In', 'Der'], ['Die', 'Dresden', 'In'], ['Die', 'Goethe', 'In'], ['Die', 'In', 'Der'], ['Die', 'Dresden', 'In'], ['Die', 'In', 'Der'], ['Die', 'In', 'ISBN'], ['Die', 'In', 'Der'], ['Die', 'Der', 'In'], ['Die', 'Bonn', 'Der'], ['Die', 'Der', 'In'], ['Die', 'In', 'ISBN'], ['Die', 'In', 'Der'], ['Die', 'In', 'D

# Evaluation

To evaluate the performance of topic modeling, googel keyword research is used. Googel is a search engine that identifies and analyzes similar keywords as topic modeling. Since googel is the most used search engine, the following section checks whether the correct Wikipedia link can be found using googel's keyword research.

The rank of the search results is used as a performance index.

In [16]:
from googlesearch import search
import time

for i in keywords_list[:1]:
    
    # Definie Search
    query = "site:de.wikipedia.org" + " ".join(i) # Wikipedia + identified Key Words

    # Get Search results
    url_results = []
    for url in search(query, num_results=20):
        url_results.append(url)
        #print(url)

    # Check if url leads to the right article and save index as performance score
    titel = "Aristoteles"
    performance_index = []
    if "https://de.wikipedia.org/wiki/"+titel in url_results:
        index = url_results.index("https://de.wikipedia.org/wiki/"+titel)
        print("Correct url has been found on the", index+1 , "search result")
        performance_index.append(index)
    else:
        index = np.nan
        performance_index.append(index)

    time.sleep(30)
    break

HTTPError: 429 Client Error: Too Many Requests for url: https://www.google.com/sorry/index?continue=https://www.google.com/search%3Fq%3Dsite%253Ade.wikipedia.orgDie%252BIn%252BDer%26num%3D22%26hl%3Den%26start%3D0&hl=en&q=EhAgAwDGLy05AJUWlaBcVj9kGN78wqMGIjC77-PoxHo7AmdYG7MJPCHMIvsf0mYDHhkQeGANGYLci2TdBfJ07DaeC5zoFdTW6VIyAXI

# Sort Documents into Topics

In [46]:
top_topics = model.top_topics(corpus)

# Average topic coherence is the sum of topic coherences of all topics, divided by the number of topics.
avg_topic_coherence = sum([t[1] for t in top_topics]) / num_topics
print('Average topic coherence: %.4f.' % avg_topic_coherence)

from pprint import pprint
pprint(top_topics)

Average topic coherence: 0.0000.
[([(0.11111111, '𝐀𝐓𝐔𝐀𝐋𝐈𝐙𝐀𝐒𝐀𝐔𝐍'),
   (0.11111111, '𝐄𝐒𝐓𝐀𝐁𝐄𝐋𝐄𝐒𝐈𝐌𝐄𝐍𝐓𝐔'),
   (0.11111111, '𝐇𝐔𝐒𝐈'),
   (0.11111111, '𝐈𝐇𝐀'),
   (0.11111111, '𝐈𝐍𝐅𝐎𝐑𝐌𝐀𝐒𝐎𝐄𝐍𝐒'),
   (0.11111111, '𝐋𝐄𝐒𝐓𝐄'),
   (0.11111111, '𝐏𝐑𝐈𝐙𝐈𝐎𝐍𝐀𝐋'),
   (0.11111111, '𝐓𝐈𝐌𝐎𝐑'),
   (0.11111111, '𝐓𝐎𝐋𝐔')],
  2.520001984545054e-10),
 ([(0.11111111, '𝐀𝐓𝐔𝐀𝐋𝐈𝐙𝐀𝐒𝐀𝐔𝐍'),
   (0.11111111, '𝐄𝐒𝐓𝐀𝐁𝐄𝐋𝐄𝐒𝐈𝐌𝐄𝐍𝐓𝐔'),
   (0.11111111, '𝐇𝐔𝐒𝐈'),
   (0.11111111, '𝐈𝐇𝐀'),
   (0.11111111, '𝐈𝐍𝐅𝐎𝐑𝐌𝐀𝐒𝐎𝐄𝐍𝐒'),
   (0.11111111, '𝐋𝐄𝐒𝐓𝐄'),
   (0.11111111, '𝐏𝐑𝐈𝐙𝐈𝐎𝐍𝐀𝐋'),
   (0.11111111, '𝐓𝐈𝐌𝐎𝐑'),
   (0.11111111, '𝐓𝐎𝐋𝐔')],
  2.520001984545054e-10),
 ([(0.11111111, '𝐀𝐓𝐔𝐀𝐋𝐈𝐙𝐀𝐒𝐀𝐔𝐍'),
   (0.11111111, '𝐄𝐒𝐓𝐀𝐁𝐄𝐋𝐄𝐒𝐈𝐌𝐄𝐍𝐓𝐔'),
   (0.11111111, '𝐇𝐔𝐒𝐈'),
   (0.11111111, '𝐈𝐇𝐀'),
   (0.11111111, '𝐈𝐍𝐅𝐎𝐑𝐌𝐀𝐒𝐎𝐄𝐍𝐒'),
   (0.11111111, '𝐋𝐄𝐒𝐓𝐄'),
   (0.11111111, '𝐏𝐑𝐈𝐙𝐈𝐎𝐍𝐀𝐋'),
   (0.11111111, '𝐓𝐈𝐌𝐎𝐑'),
   (0.11111111, '𝐓𝐎𝐋𝐔')],
  2.520001984545054e-10),
 ([(0.111111104, '𝐀𝐓𝐔𝐀𝐋𝐈𝐙𝐀𝐒𝐀𝐔𝐍'),
   (0.111111104, '𝐄𝐒𝐓𝐀𝐁𝐄𝐋𝐄𝐒𝐈𝐌𝐄𝐍𝐓𝐔'),
   (0.111111104, '𝐇𝐔𝐒𝐈'),
   (0.111111104

In [14]:
import gensim
import pyLDAvis

import pyLDAvis.gensim_models as gensimvis

pyLDAvis.enable_notebook()

gensimvis.prepare(model, corpus, dictionary)

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
7     -0.178623 -0.105556       1        1  32.738864
9     -0.164679 -0.047015       2        1  29.733686
1     -0.153652 -0.037083       3        1  16.068397
8     -0.156878 -0.014438       4        1  10.054447
6     -0.089564  0.179797       5        1   5.150998
0     -0.083684  0.008308       6        1   3.964324
2      0.033916  0.097708       7        1   1.887974
4      0.252848 -0.057895       8        1   0.401225
3      0.269985 -0.011951       9        1   0.000061
5      0.270331 -0.011875      10        1   0.000024, topic_info=       Term          Freq         Total Category  logprob  loglift
942     ref  50495.000000  50495.000000  Default  30.0000  30.0000
256     der  95982.000000  95982.000000  Default  29.0000  29.0000
271     die  80410.000000  80410.000000  Default  28.0000  28.0000
814    nbsp  21494.000000  21494.000000  Default  27.0000  27.0000
588      in  58546.000000  58546.000000  Default  26.0000  26.0000
...     ...           ...           ...      ...      ...      ...
804    name      0.000003  11016.813517  Topic10 -12.3702  -6.6965
822   nicht      0.000003   8305.461218  Topic10 -12.3702  -6.4140
1032   sich      0.000003  14771.546022  Topic10 -12.3702  -6.9897
1243  wurde      0.000003   9706.374704  Topic10 -12.3702  -6.5698
1267     zu      0.000003  18539.923680  Topic10 -12.3702  -7.2170

[822 rows x 6 columns], token_table=        Topic      Freq      Term
term                             
137638      6  0.964981  100x33px
123250      4  0.999073       1pt
117005      8  0.909172   8000ers
221550      4  0.996684   92uvres
1342        1  0.008283         _
...       ...       ...       ...
5613        3  0.042573         α
5613        4  0.063860         α
5613        5  0.856791         α
5613        6  0.005322         α
5613        7  0.015965         α

[1629 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[8, 10, 2, 9, 7, 1, 3, 5, 4, 6])